# Forecasting with t0-gamma

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/autogluon/autogluon/blob/master/docs/tutorials/timeseries/forecasting-t0.ipynb)
[![Open In SageMaker Studio Lab](https://studiolab.sagemaker.aws/studiolab.svg)](https://studiolab.sagemaker.aws/import/github/autogluon/autogluon/blob/master/docs/tutorials/timeseries/forecasting-t0.ipynb)

AutoGluon-TimeSeries supports zero-shot probabilistic forecasting with [`t0-gamma`](https://huggingface.co/theforecastingcompany/t0-gamma), an open-weights forecasting foundation model developed by [The Forecasting Company](https://theforecastingcompany.com/). T0 can use numeric past covariates and numeric covariates known over the prediction horizon.

T0 is inference-only in AutoGluon. Calling `fit()` prepares the predictor and loads the pretrained checkpoint; it does not fine-tune the model on the supplied data.

In [ ]:
!pip install -q "autogluon.timeseries[t0]"

## Zero-shot univariate forecasting

We start with a subset of the [Australian Electricity Demand dataset](https://zenodo.org/records/4659727). In this first example, T0 forecasts from only the historical target values.

In [ ]:
import pandas as pd

from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

In [ ]:
data = TimeSeriesDataFrame.from_path(
    "https://autogluon.s3.amazonaws.com/datasets/timeseries/australian_electricity_subset/test.csv"
)
data.head()

In [ ]:
prediction_length = 48
train_data, test_data = data.train_test_split(prediction_length)

Select `T0` in `hyperparameters` to load t0-gamma. The checkpoint is public, so no Hugging Face authentication is needed. The checkpoint-dependent cells in this tutorial are skipped by the automatic documentation build.

In [ ]:
predictor = TimeSeriesPredictor(prediction_length=prediction_length).fit(
    train_data,
    hyperparameters={"T0": {}},
)

In [ ]:
predictions = predictor.predict(train_data, model="T0")
predictor.plot(test_data, predictions, max_history_length=300)

## Forecasting with covariates

Next, we use an **Electrical Load Forecasting** dataset. The goal is to forecast the next 24 hours of `load` using weather variables. We assume weather forecasts are available over the prediction horizon, so `airtemperature`, `dewtemperature` and `sealvlpressure` are all known covariates.

T0 supports numeric past-only and known covariates. Other dynamic numeric columns not listed in `known_covariates_names`, if present, are treated as past covariates.

In [ ]:
data = TimeSeriesDataFrame.from_path(
    "https://autogluon.s3.amazonaws.com/datasets/timeseries/bull/test.parquet",
    id_column="id",
)
data.head()

In [ ]:
prediction_length = 24
known_covariates_names = ["airtemperature", "dewtemperature", "sealvlpressure"]
train_data, test_data = data.train_test_split(prediction_length=prediction_length)

# AutoGluon expects only the forecast-horizon portion here: [H, F] per item.
future_known_covariates = test_data.slice_by_timestep(-prediction_length, None)[known_covariates_names]

In [ ]:
predictor = TimeSeriesPredictor(
    prediction_length=prediction_length,
    target="load",
    known_covariates_names=known_covariates_names,
    eval_metric="MASE",
).fit(
    train_data,
    hyperparameters={"T0": {}},
)

In [ ]:
predictions = predictor.predict(
    train_data,
    known_covariates=future_known_covariates,
    model="T0",
)
predictor.plot(test_data, predictions, max_history_length=300)

## AutoGluon and T0 data formats

Users interact with {class}`~autogluon.timeseries.TimeSeriesDataFrame`, a long dataframe indexed by `(item_id, timestamp)`. The integration converts this dataframe into model-ready tensors internally; users do not need to import or construct `t0.TimeSeries`.

For one item, let:

- `P` be the numeric past covariates;
- `F` be the numeric known covariates;
- `T` be the historical context length;
- `H` be the prediction length.

AutoGluon stores the data as:

| Input | Dataframe layout | Variate-first view |
| --- | --- | --- |
| Historical `TimeSeriesDataFrame` | `[T, 1 + P + F]` | `[1 + P + F, T]` |
| Future `known_covariates` | `[H, F]` | `[F, H]` |

The adapter separates the columns by their role in the T0 input:

```text
target                         [1, T]     -> TARGET row
known covariates in history    [F, T]
known covariates in future     [F, H]     -> FUTURE rows        [F, T + H]
past covariates                [P, T]     -> HISTORICAL rows    [P, T] (+ H when known covariates are present)
```

The adapter fills one batch of arrays and wraps it in a T0 `TimeSeries`. Items with shorter histories are padded on the left. Past covariates use `HISTORICAL` rows and known covariates use `FUTURE` rows; only the target is forecast.

Setting `cross_learning=True` joins all items of a batch into one T0 group, so each item's target and covariate rows attend to those of the other items in the batch; the group size is `batch_size`. It works like `cross_learning` in Chronos-2 and corresponds to `cross_inference` in tfc-t0's `T0Predictor`. It is off by default, which forecasts each item on its own.

:::{note}
AutoGluon keeps a known covariate's historical values in the main `TimeSeriesDataFrame` and accepts its horizon values through the separate `known_covariates` argument to `predict()`. The adapter combines these two parts along time when constructing the batch.
:::

## Limitations

- AutoGluon's T0 integration supports numeric past and known covariates.
- Categorical covariates and static features are ignored by T0.
- T0 is currently available for inference, not fine-tuning, through AutoGluon.
- The model checkpoint must be downloaded from Hugging Face the first time it is used.

See {class}`~autogluon.timeseries.models.T0Model` for all model hyperparameters.